# Phase 1.5: Evaluation Harness

Set up train/test split and evaluation metrics.

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.cache import DataCache
from src.evaluation.harness import RecommendationHarness

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
reviews = cache.load_dataframe('reviews_cross_city')

In [ ]:
# Create evaluation harness (one held-out target city per user, split by user)
harness = RecommendationHarness(
    reviews,
    businesses,
    test_size=PHASE1_TEST_SIZE,
    random_state=RANDOM_STATE,
    n_negatives=PHASE1_NEGATIVES_PER_POSITIVE
)

train_reviews, test_reviews = harness.create_train_test_split()

print(f"Train history reviews: {len(train_reviews):,}")
print(f"Test history reviews: {len(test_reviews):,}")
print(f"Train users: {len(harness.train_users):,}")
print(f"Test users: {len(harness.test_users):,}")

In [ ]:
# Training labels: held-out city = 1, sampled unvisited cities = 0
all_cities = sorted(businesses['city'].unique())
train_labels = harness.get_training_labels(all_cities)

print(f"Training labels: {train_labels.shape}")
print(train_labels['label'].value_counts())
train_labels.head()

In [ ]:
# Test pairs + ranking candidates (all cities minus the user's history cities)
test_pairs = harness.get_test_pairs()
test_candidates = {
    user_id: harness.get_ranking_candidates(user_id, city, all_cities)
    for user_id, city in test_pairs
}

print(f"Test pairs (user, held_out_city): {len(test_pairs):,}")
print(f"Example pairs:")
for user_id, city in test_pairs[:5]:
    print(f"  User {user_id} held out {city} ({len(test_candidates[user_id])} candidates)")

In [ ]:
# Save harness outputs for baselines and Phase 4 models
cache.save_dataframe(train_reviews, 'train_reviews')
cache.save_dataframe(test_reviews, 'test_reviews')
cache.save_dataframe(train_labels, 'train_labels')
cache.save_metadata({
    'test_pairs': [(uid, city) for uid, city in test_pairs],
    'candidates': test_candidates
}, 'test_pairs')

print("✓ Evaluation harness ready")
print("Ready for Phase 1.6: Baseline Models")